# 05: E2E Baseline Pipeline

## Learning Goals | 学习目标

1. Understand the **end-to-end (E2E) pipeline** concept / 理解端到端管道的概念
2. Master **Persistence Forecast** (naive baseline) / 掌握持续法基线预测
3. Understand **P&L (Profit & Loss)** in power trading / 理解电力交易盈亏计算
4. See the full chain: **data -> forecast -> trading -> P&L** / 见证完整数据链路

## Why this is Notebook 05 (not last)? | 为什么这是第 5 个 notebook？

Intuition says "master each component first, then integrate." But research shows:

> **Pitfall #5: The "Perfect Model, No Integration" Trap**
> Learners spend weeks optimizing forecast accuracy in isolation,
> but never connect it to a simulation system.

**Solution: Run the skeleton first, then add muscle.**
Use the simplest model (persistence) to prove the entire pipeline works,
then swap in XGBoost later.

This is called a **Walking Skeleton** -- a software engineering best practice:
build the bones first, make it walk, then add flesh.

---
## Data: Shandong 15-min | 数据: 山东 15 分钟级

We use real Shandong provincial spot market data (2024-2026),
96 data points per day, including load, renewable generation, and real-time prices.

In [ ]:
# === Imports | 导入依赖 ===
from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.forecaster import (
    persistence_forecast, calculate_pnl, plot_pnl
)
import plotly.io as pio
pio.renderers.default = 'notebook'

---
## Step 1: Data Loading | 数据加载

Reuse the standard DataLoader -> Cleaner pipeline.
Same pattern as Notebook 01, but now with **Shandong 15-min data**.

**Key Insight**: This is the power of the **Contract** --
DataLoader defines a unified output format.
Downstream modules don't care where the data comes from.

核心洞察：这就是合约的力量——DataLoader 定义了统一的输出格式，下游模块不需要关心数据来源。

In [ ]:
# Load Shandong 2024 summer data (Jun-Sep) | 加载山东 2024 夏季数据
loader = create_loader("shandong")
df = loader.load_data("2024-06-01", "2024-09-30")
df = clean_data(df)

# 15-min data: 96 points/day | 15 分钟级数据: 每天 96 个点
n_days = len(df) / 96
print(f"Data ready: {len(df):,} rows ({n_days:.0f} days @ 96 pts/day)")
print(f"Date range: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
print(f"Avg load: {df['load_mw'].mean():.0f} MW")
print(f"Columns: {list(df.columns)}")
df.head()

---
## Step 2: Persistence Forecast | 持续法预测

### What is Persistence? | 持续法是什么？

$$\text{forecast}_t = \text{actual}_{t - \Delta}$$

**Use yesterday's same-moment value as today's prediction.**
用昨天同时刻的实际值作为今天的预测。

For 15-min data, Delta = 96 points (one full day).
对 15 分钟数据，Delta = 96 个点（一整天）。

### Why is this NOT stupid? | 为什么不蠢？

Electricity load has a strong **diurnal cycle**:
- Morning ramp-up (people wake up) -> Midday peak (industrial/commercial)
- Evening peak (people go home) -> Night trough (most people sleep)

Yesterday's 3:00 PM load is a good starting guess for today's 3:00 PM.
Persistence often achieves 3-5% MAPE on 15-min data (96 lag points) -- without any training!

### Mathematical intuition | 数学直觉

Load time series decomposition:
$$L(t) = T(t) + S(t) + C(t) + R(t)$$

- $T(t)$: Trend (slow -- economic growth)
- $S(t)$: Seasonal (year/week patterns)
- $C(t)$: Cyclical -- **diurnal** <- Persistence captures this!
- $R(t)$: Residual (unpredictable random component)

Persistence assumes $C(t) \approx C(t-96)$ and other components change slowly.

In [ ]:
# Execute persistence forecast | 执行持续法预测
forecast = persistence_forecast(df)

# Calculate prediction errors | 计算预测偏差
errors = (forecast - df['load_mw']).abs()
mae = errors.mean()
mape = (errors / df['load_mw']).mean() * 100

print(f"Persistence MAE:  {mae:.0f} MW")
print(f"Persistence MAPE: {mape:.2f}%")
print(f"Relative to mean load: {mae / df['load_mw'].mean() * 100:.1f}%")
print()
print("Interpretation:")
print(f"  Average prediction error is {mae:.0f} MW using only")
print(f"  yesterday's values. Not bad for a model with zero training!")
print()
print("解读: 只用"昨天同一时刻"这一个规则，平均预测偏差")
print(f"  {mae:.0f} MW。为零训练成本的基线模型来说还可以。")

---
## Step 3: P&L Calculation | 盈亏计算

### Simulated Trading Model | 模拟交易模型

Imagine you are a **power retailer**:
1. You predict tomorrow's load for each 15-min interval
2. Buy electricity in the day-ahead market at your predicted volume
3. At delivery time, settle at actual load

**Simplified P&L Formula:**
$$\text{P\&L}_t = -|\text{forecast}_t - \text{actual}_t| \times \frac{\text{price}}{1000}$$

Why negative? Every deviation = a "penalty" in this simplified model.
In real markets, small deviations are tolerated within a deadband.

> This is a highly simplified learning model.
> Real power markets have: imbalance bands, ancillary services,
> congestion management, nodal pricing, and dozens of mechanisms.

简化模型说明：偏差的绝对值乘以电价就是每时刻的"损失"。
预测偏差越大，亏得越多。P&L 永远是负数（简化假设下）。

In [ ]:
# Compute cumulative P&L | 计算累计盈亏
# Using 50 RMB/MWh as reference price (approx Chinese spot market average)
pnl = calculate_pnl(df['load_mw'], forecast, price_per_mwh=50)

print(f"Cumulative P&L: {pnl.iloc[-1]:.2f} RMB")
print()
print("Interpretation | 解读:")
print("  P&L is negative -- the persistence forecast has deviations,")
print("  producing 'trading losses'. This is completely normal!")
print("  Persistence is just the baseline. The goal is:")
print("    XGBoost/Lear P&L >> Persistence P&L")
print()
if pnl.iloc[-1] > pnl.iloc[0]:
    print("  P&L trending UP -- forecast errors are shrinking")
    print("  P&L 在上升 -- 预测偏差在减小")
else:
    print("  P&L trending DOWN -- forecast errors are compounding")
    print("  P&L 在下行 -- 预测偏差在累积")

---
## Step 4: End-to-End Visualization | 端到端可视化

Integrate data -> forecast -> P&L into a single interactive chart.

This is the **Phase 1 final output** -- proving the entire pipeline works.

这是 Phase 1 的最终产出 —— 证明整个管道跑通了。

In [ ]:
# Plot end-to-end result | 绘制端到端结果图
# Show first 14 days for readability | 显示前 14 天以便阅读
n_pts = 96 * 14  # 14 days of 15-min data
df_sample = df.iloc[:n_pts]
forecast_sample = forecast.iloc[:n_pts]
pnl_sample = pnl.iloc[:n_pts]

fig = plot_pnl(
    df_sample,
    forecast_sample,
    pnl_sample,
    title="E2E Baseline -- Persistence Prediction vs Actual (Shandong, First 14 Days)"
)
fig.show()

---
## Step 5: Zoom into a Single Day | 单日放大观察

With 15-min data, we can see intra-day patterns clearly.
15 分钟级数据让我们能清晰看到日内负荷模式。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pick a single day (e.g., day 3 = index 192 to 288) | 选一天放大
start, end = 192, 288
day_actual = df['load_mw'].iloc[start:end]
day_forecast = forecast.iloc[start:end]
day_ts = df['timestamp'].iloc[start:end]

fig_day = go.Figure()
fig_day.add_trace(go.Scatter(
    x=day_ts, y=day_actual, mode='lines+markers',
    name='Actual Load | 实际负荷',
    line=dict(color='#1f77b4', width=2.5)
))
fig_day.add_trace(go.Scatter(
    x=day_ts, y=day_forecast, mode='lines+markers',
    name='Persistence (t-1d) | 持续法',
    line=dict(color='#ff7f0e', width=1.5, dash='dash')
))

fig_day.update_layout(
    title=f"Single Day Detail: {day_ts.iloc[0].date()} (15-min resolution)",
    xaxis_title="Time", yaxis_title="Load (MW)",
    hovermode="x unified", height=400
)
fig_day.show()

---
## Congratulations! | 恭喜！

You just completed your first end-to-end pipeline with **Shandong 15-min data**:

```
Shandong Fetcher -> DataLoader -> Cleaner
  -> Persistence Forecast -> P&L -> Plotly Visualization
```

This is the **Walking Skeleton** -- bones in place, ready for muscle
(XGBoost/Lear replacing persistence).

## Key Takeaways | 学习笔记

1. E2E pipeline runs: data -> forecast -> P&L -> chart
2. Persistence as baseline -- simple but effective (captures diurnal cycle)
3. P&L quantifies "good forecast -> good trading"
4. 15-min data reveals intra-day patterns impossible to see with yearly data
5.  Next: Notebook 06 -> LEAR price forecasting

## Reflection Questions | 反思题

1. If you had to beat persistence with ONE simple rule, what would you add?
2. Why is P&L always negative? Under what real-market conditions could it be positive?
3. Look at the single-day chart: which hours does persistence fail most? Why?
4. How would you adapt the pipeline for real trading decisions?

思考：如果要打败持续法，你会加什么规则？周中和周末的负荷模式有何不同？